***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节： [9.15 连续谱源表生成](9_15_continuum_source_catalogs.ipynb)
    * 下一节： [9.17 QA 与失败模式识别](9_17_quality_control_failure_modes.ipynb)

***


## 9.16 成像参数选择：从科学目标到 QA 闭环

成像参数不是软件界面里的技术细节，而是把科学目标翻译成图像产品的决策。相同的可见度数据，用不同的 `cell`、`imsize`、weighting、uv taper、mask、`niter` 和 threshold，可能得到不同的角分辨率、噪声、旁瓣、扩展通量恢复和源表可靠性。因此，成像参数不能只按默认值处理，也不能只用“图像看起来更清楚”来判断。

本节把第 5 章关于采样、PSF、权重和宽场效应的原理，落到一个实践案例中。目标场包含一个明亮紧致源、一个弱扩展源和视场边缘的离轴源。这样的场景很常见：若科学目标是最深点源探测，参数选择会偏向灵敏度；若目标是扩展通量，参数选择会偏向短基线和低表面亮度结构；若目标是高分辨率形态，则需要更重视长基线、像元采样和旁瓣控制。


### 9.16.1 先定义科学目标，再选择参数

一个稳健的成像流程应从科学问题开始，而不是从 `tclean` 或 WSClean 的参数列表开始。点源巡天最关心离源 RMS、源表可靠性和合成波束是否足够稳定；星系扩展结构研究更关心低表面亮度通量、最大可恢复尺度和 uv taper；高动态范围图像则更关心亮源旁瓣、方向相关误差和 mask 策略。不同目标要求不同的图像产品，不存在一组参数同时最优。

下面的决策树给出一条实用顺序：先由科学目标决定视场和分辨率需求，再选择 cell size 和 image size；随后根据噪声、分辨率和扩展结构需求选择 weighting 与 taper；最后通过 mask、threshold 和 QA loop 检查结果是否可信。


![成像参数选择决策树](figures/practical_imaging_parameter_decision_tree.png)

**图 9.16.1** 成像参数选择决策树。参数选择应从科学目标出发，并用 PSF、残差、通量稳定性和负源检查闭环验证。若 QA 不通过，应回到权重、taper、mask 或校准诊断，而不是只继续加深 CLEAN。


### 9.16.2 Cell size 与 image size：先保证图像网格不误导测量

Cell size 决定图像像素对合成波束的采样。若像元过大，合成波束只有很少像素覆盖，峰值位置、峰值通量、源大小和高斯拟合都会产生离散化偏差；若像元过小，图像文件和计算代价会显著增加，却不一定提供新的独立信息。实践上，合成波束长轴通常至少需要约 3--5 个像素采样。这个经验规则的本质，是让图像网格足够细，可以稳定表示恢复波束和源形态。

Image size 决定成像区域。若图像只覆盖中心科学目标，视场边缘或主波束外的亮源可能没有被 deconvolve，却仍会通过旁瓣污染目标区域；若图像太大，计算代价和宽场误差处理需求都会增加。对于宽场或含亮离轴源的观测，image size 应同时考虑科学目标、primary beam、outlier field 和后续源表范围。


![Cell size 与 image size 的典型选择问题](figures/practical_imaging_cell_imsize_choices.png)

**图 9.16.2** Cell size 与 image size 的选择。过粗像元会低估形态与位置细节；过小图像会漏掉离轴亮源或造成边缘去卷积问题；合适图像区域应覆盖科学目标及会产生显著旁瓣的离轴结构。


### 9.16.3 Weighting：噪声、分辨率和旁瓣的取舍

第 5.4 节已经说明，权重函数 $W(u,v)$ 会直接改变 dirty beam：

$$
B_D(l,m)=\mathcal{F}^{-1}\{W(u,v)\}.
$$

Natural weighting 通常给予密集采样区域更大权重，热噪声较低，对短基线和扩展结构较友好，但角分辨率较差。Uniform weighting 强调稀疏采样区域，常提高分辨率，却会增加噪声并抬高旁瓣。Briggs robust weighting 位于二者之间，通过一个连续参数调节噪声与分辨率的平衡。

因此，weighting 的选择不应只看合成波束大小。对于弱点源巡天，较低 RMS 可能比最高分辨率更重要；对于混杂场中的相邻源分离，较高分辨率可能值得牺牲部分灵敏度；对于扩展源通量，自然权重或 taper 可能比 uniform 更合理。


![不同 weighting 对 PSF 与 dirty image 的影响](figures/practical_imaging_weighting_tradeoff_case.png)

**图 9.16.3** Natural、Briggs 和 uniform weighting 的对比。Natural 权重通常较深但分辨率较低；uniform 权重分辨率较高但噪声和旁瓣风险更大；Briggs 权重提供中间取舍。


### 9.16.4 UV taper、mask 与 threshold：保护扩展通量，也保护残差统计

UV taper 会降低长基线的相对权重，使图像分辨率变差，但对低表面亮度扩展结构更友好。它不是“让图像变模糊”的后处理，而是在 $uv$ 空间改变测量权重。若科学目标是扩展通量或弥散结构，taper 后的图像往往更接近所需产品；若科学目标是紧致结构位置和分辨率，过强 taper 会丢失形态信息。

Mask 和 threshold 控制 CLEAN 在哪里建模、建模到多深。Mask 过紧会把真实源翼部或扩展通量留在残差中，导致通量低估；mask 过松或 threshold 过深会把噪声和残余旁瓣吸入模型，造成 clean bias 或伪结构。稳健做法不是追求残差越低越好，而是看残差是否接近噪声、源通量是否随清理深度稳定、负源统计是否合理，以及模型是否仍有物理解释。


![UV taper、mask 与 threshold 的成像后果](figures/practical_imaging_taper_mask_threshold.png)

**图 9.16.4** UV taper 和 CLEAN 控制的典型后果。Taper 可以改善扩展低面亮度结构的恢复，但会牺牲分辨率；mask 过紧会留下源通量，mask 过松或清理过深会把噪声纳入模型。


### 9.16.5 参数选择需要 QA 表，而不只需要最终图

一个可复现的成像案例，应保留参数、诊断图和判断理由。最低限度的 QA 应包括：合成波束是否被充分采样；图像范围是否覆盖会影响目标区域的亮源；PSF 旁瓣是否可接受；残差图是否接近噪声；目标源通量是否随 weighting、taper 或 CLEAN 深度稳定；负源数量是否与噪声预期一致；源表生成是否对阈值变化过于敏感。

这种 QA 表的意义，是把成像参数从“个人经验”转化为可检查的教学对象。不同软件的参数名称不同，但判断对象基本相同：图像网格、采样权重、去卷积区域、停止条件、残差统计和科学量稳定性。


![成像参数 QA 表](figures/practical_imaging_parameter_qa_sheet.png)

**图 9.16.5** 一个最小成像参数 QA 表。它不是替代经验，而是把经验显式化：每个参数都应对应一个判断标准和一个可能失败模式。


### 9.16.6 与真实软件工作流的对应

在 CASA 的 `tclean` 中，这一节对应的关键参数包括 `cell`、`imsize`、`gridder`、`weighting`、`robust`、`uvtaper`、`niter`、`threshold`、`usemask`、`pbcor` 和宽场/宽带相关选项。在 WSClean 中，对应参数名称不同，但仍围绕像元、图像大小、权重、taper、多尺度清理、阈值、mask 和主波束校正组织。教学时不宜让参数表压过判断链：软件参数只是表达这些判断的入口。

实际处理建议至少产出三组图像：一个偏灵敏度的 natural/robust 图像，一个偏分辨率的 robust/uniform 图像，一个针对扩展结构的 tapered 图像。若三组图像中的科学结论一致，可信度会显著提高；若结论随参数强烈变化，说明科学量仍受成像选择支配，需要回到 $uv$ 覆盖、校准、mask、短间距或噪声模型重新检查。


### 9.16.7 本节结论

成像参数选择的核心不是记住默认值，而是理解每个参数改变了哪一部分测量问题。Cell size 和 image size 决定图像网格是否足够表达数据；weighting 和 taper 决定哪些空间频率被强调；mask、threshold 和 `niter` 决定模型如何从 dirty image 中被提取；QA loop 决定图像是否足以支撑科学测量。

第 9.14 节的端到端案例、第 9.15 节的源表案例和本节的参数决策树应放在一起使用：先得到可信图像，再生成可信测量和源表。没有参数 QA 的图像，即使外观漂亮，也不应直接进入科学解释。

***
